## Neural flow fields for streamlines

* Learning: tiny CNNs, vector fields, regularization, sampling.
* Idea: a small CNN outputs a 2D vector field u(x, y). You integrate streamlines to get long continuous curves. Train with a loss that matches a target image or a text embedding and add curvature and spacing penalties.
* Why plotter friendly: long continuous lines, controllable density, few pen lifts.

In [11]:
# CLIP-guided single-width diffvg line art
import random
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import torch
import torchvision.transforms as T

import clip
import diffvg  # noqa: F401
import pydiffvg

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL, _ = clip.load("ViT-B/32", device=DEVICE, jit=False)
MODEL = MODEL.eval()

BASE_AUG = [
    T.RandomPerspective(fill=1, p=0.8, distortion_scale=0.4),
    T.ColorJitter(hue=0.02, saturation=0.1, brightness=0.05),
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
]
CLIP_NORMALIZE = T.Normalize(
    mean=(0.48145466, 0.4578275, 0.40821073),
    std=(0.26862954, 0.26130258, 0.27577711),
)


def _build_augmentations(num_augs: int, use_clip_normalize: bool) -> List[T.Compose]:
    aug_layers: List[T.Compose] = []
    for _ in range(num_augs):
        ops = list(BASE_AUG)
        if use_clip_normalize:
            ops.append(CLIP_NORMALIZE)
        aug_layers.append(T.Compose(ops))
    return aug_layers


def _init_scene(
    *,
    num_paths: int,
    segment_range: Sequence[int],
    stroke_width: float,
    canvas_width: int,
    canvas_height: int,
    rng: random.Random,
    line_color: Sequence[float],
) -> Tuple[List[pydiffvg.Path], List[pydiffvg.ShapeGroup]]:
    min_segments, max_segments = segment_range
    shapes: List[pydiffvg.Path] = []
    groups: List[pydiffvg.ShapeGroup] = []
    radius = 0.20 * min(canvas_width, canvas_height)

    for _ in range(num_paths):
        seg_count = rng.randint(min_segments, max_segments)
        num_control_points = torch.full((seg_count,), 2, dtype=torch.int32)
        px, py = rng.random() * canvas_width, rng.random() * canvas_height
        points = [(px, py)]
        for _ in range(seg_count):
            for _ in range(3):
                px += rng.gauss(0.0, radius)
                py += rng.gauss(0.0, radius)
                points.append((px, py))
        pts_tensor = torch.tensor(points, dtype=torch.float32)
        path = pydiffvg.Path(
            num_control_points=num_control_points,
            points=pts_tensor,
            stroke_width=torch.tensor(float(stroke_width)),
            is_closed=False,
        )
        shapes.append(path)
        color = torch.tensor(line_color, dtype=torch.float32)
        group = pydiffvg.ShapeGroup(
            shape_ids=torch.tensor([len(shapes) - 1]),
            fill_color=None,
            stroke_color=color,
        )
        groups.append(group)
    return shapes, groups


def _render_to_tensor(
    canvas_width: int,
    canvas_height: int,
    shapes: Sequence[pydiffvg.Path],
    groups: Sequence[pydiffvg.ShapeGroup],
    *,
    samples: int,
    seed: int,
) -> torch.Tensor:
    scene_args = pydiffvg.RenderFunction.serialize_scene(
        canvas_width,
        canvas_height,
        list(shapes),
        list(groups),
    )
    render = pydiffvg.RenderFunction.apply
    img = render(canvas_width, canvas_height, samples, samples, seed, None, *scene_args)
    rgb = img[:, :, 3:4] * img[:, :, :3] + (1 - img[:, :, 3:4])
    rgb = rgb[:, :, :3].permute(2, 0, 1).unsqueeze(0)
    return rgb


def _curvature_loss(paths: Sequence[pydiffvg.Path]) -> torch.Tensor:
    loss = torch.tensor(0.0, device=DEVICE)
    for path in paths:
        pts = path.points.view(-1, 2)
        if pts.shape[0] < 3:
            continue
        forward = pts[1:] - pts[:-1]
        second = forward[1:] - forward[:-1]
        loss = loss + second.pow(2).sum()
    return loss / max(len(paths), 1)


def run_single_width_clipdraw(
    prompt: str,
    *,
    canvas_size: int = 384,
    num_paths: int = 80,
    segment_range: Sequence[int] = (1, 3),
    stroke_width: float = 2.2,
    num_iter: int = 280,
    point_lr: float = 1.0,
    curve_weight: float = 5e-3,
    entropy_weight: float = 1e-3,
    guidance_scale: float = 1.0,
    use_clip_normalize: bool = False,
    num_augs: int = 4,
    samples: int = 2,
    save_every: int = 80,
    line_color: Sequence[float] = (0.05, 0.05, 0.05, 1.0),
    out_dir: Path | str = Path("artifacts/clipdraw"),
    seed: int | None = 42,
) -> Dict[str, str]:
    """Optimize a bundle of single-width Bézier strokes to match a text prompt."""
    if isinstance(out_dir, str):
        out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if seed is not None:
        random.seed(seed)
        torch.manual_seed(seed)

    pydiffvg.set_use_gpu(DEVICE.type == "cuda")
    pydiffvg.set_device(DEVICE)

    shapes, groups = _init_scene(
        num_paths=num_paths,
        segment_range=segment_range,
        stroke_width=stroke_width,
        canvas_width=canvas_size,
        canvas_height=canvas_size,
        rng=random,
        line_color=line_color,
    )

    point_params: List[torch.Tensor] = []
    for path in shapes:
        path.points.requires_grad = True
        point_params.append(path.points)
    for group in groups:
        group.stroke_color.requires_grad = False

    optim = torch.optim.Adam(point_params, lr=point_lr)
    augmenters = _build_augmentations(num_augs, use_clip_normalize)

    tokens = clip.tokenize(prompt).to(DEVICE)
    with torch.no_grad():
        text_features = MODEL.encode_text(tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    final_png = out_dir / "final.png"
    final_svg = out_dir / "final.svg"

    for step in range(num_iter):
        optim.zero_grad()
        img = _render_to_tensor(canvas_size, canvas_size, shapes, groups, samples=samples, seed=step)
        img = img.clamp(0.0, 1.0).to(DEVICE)

        augmented = [aug(img) for aug in augmenters]
        batch = torch.cat(augmented, dim=0)
        image_features = MODEL.encode_image(batch)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        clip_term = -torch.cosine_similarity(image_features, text_features, dim=-1)
        clip_loss = guidance_scale * clip_term.mean()

        curvature = _curvature_loss(shapes)
        centers = torch.stack([path.points.view(-1, 2).mean(dim=0) for path in shapes])
        spread = centers.var(dim=0).sum()
        entropy_loss = torch.relu(1.4 - spread)

        loss = clip_loss + curve_weight * curvature + entropy_weight * entropy_loss
        loss.backward()
        optim.step()

        for path in shapes:
            path.points.data[:, 0].clamp_(0.0, canvas_size)
            path.points.data[:, 1].clamp_(0.0, canvas_size)

        if save_every and (step % save_every == 0 or step == num_iter - 1):
            with torch.no_grad():
                preview = _render_to_tensor(canvas_size, canvas_size, shapes, groups, samples=samples, seed=step)
            out_path = out_dir / f"iter_{step:04d}.png"
            pydiffvg.imwrite(preview[0].permute(1, 2, 0).cpu(), str(out_path), gamma=1.0)

    with torch.no_grad():
        final_img = _render_to_tensor(canvas_size, canvas_size, shapes, groups, samples=samples, seed=num_iter)
    pydiffvg.imwrite(final_img[0].permute(1, 2, 0).cpu(), str(final_png), gamma=1.0)
    pydiffvg.save_svg(str(final_svg), canvas_size, canvas_size, shapes, groups)

    return {"png": str(final_png), "svg": str(final_svg)}


ModuleNotFoundError: No module named 'diffvg'

In [10]:
# Minimal quickstart: generate a single piece of line art.
from IPython.display import Image

result = run_single_width_clipdraw(
    "abstract single-line drawing of a windswept tree",
    seed=9,
)
Image(filename=result["png"])


NameError: name 'run_single_width_clipdraw' is not defined